In [30]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from typing import Annotated
from dotenv import load_dotenv
import os 
from langchain_groq import ChatGroq


load_dotenv()

True

In [31]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [32]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

In [33]:
class State(TypedDict):
    messages:Annotated[list, add_messages]

In [34]:
from langchain_core.messages import SystemMessage, HumanMessage

sys_msg = SystemMessage(content="You are a helpful assistant that converts natural language to SQL queries. only return sql only")

def texttosql(state:State):
    return {"messages":llm.invoke([sys_msg]+state["messages"])}

In [35]:
workflow = StateGraph(State)

workflow.add_node("texttosql", texttosql)

workflow.add_edge(START, "texttosql")
workflow.add_edge("texttosql", END)

In [36]:
graph  = workflow.compile()

messages = [HumanMessage(content="What is the total revenue for last month?")]

messages = graph.invoke({"messages":messages})

for m in messages["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is the total revenue for last month?
================================== Ai Message ==================================

```sql
SELECT SUM(amount) AS total_revenue
FROM transactions
WHERE transaction_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')
  AND transaction_date < DATE_TRUNC('month', CURRENT_DATE);
```
